# TabFM (Google) — Classification

Demonstrates **TabFM**, Google's zero-shot tabular foundation model, on the shared
retail/CPG classification datasets. TabFM is **self-hosted**: model weights are
downloaded from Hugging Face and inference runs locally in a single forward pass.

> **License note:** TabFM is released under a **non-commercial** license. This notebook
> is for evaluation only. See [`../README.md`](../README.md).

**Compute:** GPU cluster recommended (weights run on GPU). CPU works but is slow.

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.


In [ ]:
%pip install tabfm[pytorch] scikit-learn pandas matplotlib mlflow --quiet

In [ ]:
dbutils.library.restartPython()

## Configuration

Catalog/schema must match the shared data-prep notebook. `common/` is added to the
path so we can import the shared evaluation helpers.


In [ ]:
import os, sys

# Make the repo-level common/ package importable.
# Adjust REPO_ROOT if your repo is checked out at a different workspace path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG

# Configure catalog and schema (must match shared/notebooks/00_data_preparation, imported from common/config.py)
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# MLflow experiment (shared naming convention across vendors)
current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"common/ path:   {COMMON_PATH}  (exists={os.path.isdir(COMMON_PATH)})")

## Load the TabFM model

TabFM uses a two-step load: load the pretrained weights for the task type, then wrap
them in the sklearn-style estimator. Weights download from Hugging Face on first use.


In [ ]:
import numpy as np
import pandas as pd
import torch
import mlflow

from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
from evaluation import (
    split_xy, classification_metrics, train_baselines_classification,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# TabFM runs a transformer forward pass — it MUST be on GPU, or inference on even
# a few hundred rows takes tens of minutes. Load the pretrained weights onto CUDA
# (falls back to CPU only if no GPU is present, with a warning).
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no GPU detected — TabFM inference on CPU is extremely slow. "
          "Attach a GPU cluster (e.g. g5.xlarge).")
tabfm_clf_model = tabfm_v1_0_0.load(device=device)
print(f"TabFM classification model loaded on {device}.")

## Helper: evaluate one classification task

Runs TabFM + the shared sklearn baselines on the same split, logs both to MLflow and
to the shared Delta results table (`vendor_benchmark_results`) that the comparison
notebook reads.


In [ ]:
def evaluate_classification(table_name, target, problem_type, task_name,
                            test_size=0.2, stratify=True):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(
        df, target=target, test_size=test_size, stratify=stratify
    )
    n_train, n_features, n_test = len(X_train), X_train.shape[1], len(X_test)

    # --- TabFM (native two-step API) ---
    # n_estimators=1 keeps first-run latency low; raise for a small accuracy bump.
    with mlflow.start_run(run_name=f"{task_name}_tabfm"):
        mlflow.log_params({
            "vendor": "tabfm", "model_type": "TabFMClassifier",
            "task": task_name, "problem_type": problem_type,
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
        })
        clf = TabFMClassifier(model=tabfm_clf_model, n_estimators=1)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)
        metrics = classification_metrics(y_test, y_pred, y_proba)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="tabfm", task=task_name, problem_type=problem_type,
                   model_name="TabFMClassifier", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)
    print(f"[{task_name}] TabFM: acc={metrics['accuracy']:.4f} "
          f"f1={metrics['f1']:.4f} roc_auc={metrics['roc_auc']}")

    # --- Shared baselines (identical split) ---
    for name, m in train_baselines_classification(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type=problem_type,
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: acc={m['accuracy']:.4f} f1={m['f1']:.4f}")
    return metrics

## Binary classification — Supplier Delay Risk


In [ ]:
_ = evaluate_classification(
    table_name="supplier_delay_risk_train",
    target="is_delayed",
    problem_type="binary_classification",
    task_name="supplier_delay_risk",
    test_size=0.2, stratify=True,
)

## Multi-class classification — Material Shortage

TabFM supports classification with up to 10 classes; these tasks have 3.


In [ ]:
_ = evaluate_classification(
    table_name="material_shortage_train",
    target="shortage_risk",
    problem_type="multiclass_classification",
    task_name="material_shortage",
    test_size=0.3, stratify=True,
)

## Multi-class classification — OTIF Risk


In [ ]:
_ = evaluate_classification(
    table_name="otif_risk_train",
    target="otif_risk",
    problem_type="multiclass_classification",
    task_name="otif_risk",
    test_size=0.3, stratify=True,
)

## Results

All rows for this run are in the shared results table. The comparison notebook
aggregates across vendors.


In [ ]:
display(
    spark.table(RESULTS_TABLE)
         .where("problem_type LIKE '%classification%'")
         .orderBy("task", "vendor", "model_name")
)